# 04 — Backtest Report

*Spec section 7.8 (backtesting & evaluation, building on 7.1-7.7).*

This notebook runs the **leave-one-tournament-out** validation protocol for the minimum
viable model and reports:

- per-tournament metrics (RPS, log loss, Brier) vs. the spec targets (RPS < 0.21, Brier < 0.22, log loss < 1.02),
- calibration curves (target: tight diagonal fit, spec 7.2),
- a small ablation (with/without market features, spec 7.4),
- a sketch of contextual-layer attribution and proxy-vs-real feature comparison (spec 7.7-7.8, fully realized post-tournament).

It is tolerant of missing data: if the normalized `matches` table has not been ingested
yet, it falls back to a small synthetic match history so the narrative still runs end to
end. The real path is shown first and clearly labelled.

## Imports

Everything comes from the real `polymbappe` evaluation modules:
`run_leave_one_tournament_out` + `DEFAULT_TOURNAMENTS` (spec 7.1) and the metric helpers
(spec 7.2).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from polymbappe.config import Settings
from polymbappe.eval.backtest import (
    DEFAULT_TOURNAMENTS,
    run_leave_one_tournament_out,
    select_fixtures,
)
from polymbappe.eval.metrics import calibration_curve

RPS_TARGET, BRIER_TARGET, LOGLOSS_TARGET = 0.21, 0.22, 1.02  # spec 7.2 benchmarks
print(f"{len(DEFAULT_TOURNAMENTS)} tournaments in the LOTO set")
for t in DEFAULT_TOURNAMENTS:
    print(f"  {t.name:7s} {t.competition:16s} {t.start} -> {t.end}")

## Load match history

The real backtest reads the normalized `matches` table (and `market_odds` if ingested) from
`data/processed`. We wrap the load in a guard: if the table is missing we synthesize a
minimal multi-tournament history (two World Cups + history before each) so the protocol —
which needs >=2 tournaments with fixtures and prior history — can still execute. A real run
would skip the synthetic branch entirely.

In [ ]:
def _load_matches() -> tuple[pl.DataFrame, pl.DataFrame | None, bool]:
    """Return (matches, market_odds, is_real). Falls back to synthetic data if absent."""
    try:
        from polymbappe.data.store import read_table, table_exists
        from polymbappe.data.tables import Table

        settings = Settings()
        if table_exists(Table.MATCHES, settings):
            matches = read_table(Table.MATCHES, settings)
            odds = (
                read_table(Table.MARKET_ODDS, settings)
                if table_exists(Table.MARKET_ODDS, settings)
                else None
            )
            return matches, odds, True
    except Exception as exc:  # pragma: no cover - notebook resilience
        print(f"Real data unavailable ({exc!r}); using synthetic fallback.")
    return _synthetic_matches(), None, False


def _synthetic_matches(n_teams: int = 16, seed: int = 11) -> pl.DataFrame:
    """Tiny synthetic history covering two World Cups plus pre-tournament friendlies."""
    from datetime import date, timedelta

    rng = np.random.default_rng(seed)
    teams = [f"T{i:02d}" for i in range(n_teams)]
    strength = {t: rng.normal(0, 0.4) for t in teams}
    rows: list[dict] = []
    mid = 0

    def play(d, comp, neutral, group=None, knockout=False):
        nonlocal mid
        h, a = rng.choice(teams, size=2, replace=False)
        lam = np.exp(0.2 + strength[h] - strength[a] + (0.0 if neutral else 0.15))
        mu = np.exp(0.1 + strength[a] - strength[h])
        rows.append({
            "match_id": f"m{mid:04d}", "date": d, "home_team": h, "away_team": a,
            "home_goals": int(rng.poisson(lam)), "away_goals": int(rng.poisson(mu)),
            "competition": comp, "is_knockout": knockout,
            "neutral_site": neutral, "group": group,
        })
        mid += 1

    # WC2018 + WC2022: history before each, then the tournament fixtures themselves.
    for start in (date(2018, 6, 14), date(2022, 11, 20)):
        for k in range(120):  # pre-tournament friendlies/qualifiers (training history)
            play(start - timedelta(days=30 + k * 5), "Friendly", neutral=False)
        for k in range(48):  # tournament fixtures
            play(start + timedelta(days=k // 4), "FIFA World Cup", neutral=True, group="A")
    return pl.DataFrame(rows)


matches, market_odds, is_real = _load_matches()
print(f"{'REAL' if is_real else 'SYNTHETIC'} data: {matches.height} matches, "
      f"market_odds={'yes' if market_odds is not None else 'no'}")
matches.head()

## Run the leave-one-tournament-out backtest

`run_leave_one_tournament_out` precomputes base probabilities (Dixon-Coles + Elo + market)
once per tournament, then fits the meta-learner on every *other* tournament's frame and
evaluates on the held-out one — true out-of-fold stacking (spec 7.1). We restrict to
tournaments actually present in the data so the synthetic fallback (which only has the two
World Cups) still satisfies the >=2-tournament requirement.

In [ ]:
present = tuple(
    t for t in DEFAULT_TOURNAMENTS if not select_fixtures(matches, t).is_empty()
)
print("Tournaments with fixtures present:", [t.name for t in present])

result = run_leave_one_tournament_out(matches, present, market_odds=market_odds)
report = result.to_frame().sort("tournament")
print("Feature columns used:", result.feature_columns)
print(f"Mean RPS: {result.mean_rps:.4f}  (target < {RPS_TARGET})")
report

## Per-tournament metrics vs. targets

Bar chart of RPS per tournament against the spec target line. The spec gates every
component on beating the MVM baseline; this is the baseline being measured first (spec 7.3).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(report["tournament"], report["rps"], color="steelblue")
ax.axhline(RPS_TARGET, color="crimson", ls="--", label=f"RPS target = {RPS_TARGET}")
ax.set_ylabel("RPS")
ax.set_title("Leave-one-tournament-out RPS (lower is better)")
ax.legend()
plt.tight_layout()
plt.show()

summary = report.select([
    pl.col("rps").mean().alias("mean_rps"),
    pl.col("brier").mean().alias("mean_brier"),
    pl.col("log_loss").mean().alias("mean_log_loss"),
])
print(summary)
print("Within target?",
      {"rps": summary["mean_rps"][0] < RPS_TARGET,
       "brier": summary["mean_brier"][0] < BRIER_TARGET,
       "log_loss": summary["mean_log_loss"][0] < LOGLOSS_TARGET})

## Calibration curve

Calibration (spec 7.2) bins predicted probabilities and compares the mean prediction in
each bin to the empirical outcome frequency. A well-calibrated model hugs the diagonal. We
rebuild held-out base probabilities for the home-win outcome to feed `calibration_curve`.

In [ ]:
from polymbappe.eval.base_probs import BaseProbConfig, compute_tournament_base_probs
from polymbappe.models.meta import OUTCOMES

# Collect held-out home-win predictions vs. realized home wins across all folds.
pred_home, true_home = [], []
for t in present:
    fixtures = select_fixtures(matches, t)
    history = matches.filter(pl.col("date") < t.start)
    if history.is_empty():
        continue
    frame = compute_tournament_base_probs(
        history, fixtures, tournament=t.name, config=BaseProbConfig(), market_odds=market_odds
    )
    pred_home.extend(frame["dc_home"].to_list())
    true_home.extend((frame["label"] == OUTCOMES[0]).cast(pl.Int8).to_list())

cal = calibration_curve(np.array(true_home, float), np.array(pred_home, float), n_bins=10)
cal_valid = cal.filter(pl.col("count") > 0)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], color="grey", ls="--", label="perfect")
ax.plot(cal_valid["mean_pred"], cal_valid["empirical"], "o-", color="darkorange", label="model")
ax.set_xlabel("mean predicted P(home win)")
ax.set_ylabel("empirical frequency")
ax.set_title("Calibration — home-win outcome")
ax.legend()
plt.tight_layout()
plt.show()
cal_valid

## Ablation: with vs. without market features

The spec ablation (7.4) measures the marginal value of each input. The clearest one
available to the MVM is *market features on/off* — the calibration pipeline uses market
odds; the edge pipeline is market-blind (spec 3.6). We rerun the backtest forcing
`market_odds=None` and compare mean RPS. (With the synthetic fallback there are no market
odds, so both runs coincide — the real data path shows the true delta.)

In [ ]:
with_market = run_leave_one_tournament_out(matches, present, market_odds=market_odds)
blind = run_leave_one_tournament_out(matches, present, market_odds=None)

ablation = pl.DataFrame({
    "pipeline": ["calibration (with market)", "edge (market-blind)"],
    "mean_rps": [with_market.mean_rps, blind.mean_rps],
    "features": [", ".join(with_market.feature_columns), ", ".join(blind.feature_columns)],
})
delta = blind.mean_rps - with_market.mean_rps
print(f"Market information value (RPS): {delta:+.4f} "
      f"(>0 means market features help; spec kill-criterion threshold is 0.003)")
ablation

## Contextual-layer attribution & proxy-vs-real (post-tournament)

Spec 7.7-7.8: the contextual adjuster is trained on residuals; its **attribution** breaks
down how much each feature group (PPDA, cohesion, manager pedigree, fatigue, xG
overperformance, draw pressure) shifted each team's probabilities, and the **proxy-vs-real**
comparison (Tier B Reddit/news sentiment vs. their backtestable proxies) is only resolvable
*after* the 2026 tournament completes.

The real attribution table is `data/outputs/contextual_attribution.parquet` (spec 11). Here
we load it if present, otherwise sketch the schema with an illustrative example so the
report structure is complete. Each contextual feature is bounded to +/-3pp shift (spec 7.7).

In [ ]:
settings = Settings()
attr_path = settings.outputs_data_dir / "contextual_attribution.parquet"
if attr_path.exists():
    attribution = pl.read_parquet(attr_path)
    print("Loaded real contextual attribution.")
else:
    print("No attribution artifact yet — showing illustrative schema (spec 7.8 / 11).")
    attribution = pl.DataFrame({
        "team": ["Brazil", "Brazil", "Brazil", "Germany", "Germany"],
        "factor": ["manager_pedigree", "fatigue", "xg_overperformance", "ppda", "draw_pressure"],
        "prob_shift_pp": [0.8, -0.3, 0.5, -0.6, 0.2],  # bounded to +/-3pp
    })

proxy_vs_real = pl.DataFrame({
    "feature": ["reddit_sentiment", "news_sentiment"],
    "proxy": ["xg_overperformance", "xg_overperformance"],
    "tier": ["B (forward-test only)", "B (forward-test only)"],
    "rps_delta_vs_proxy": [None, None],  # filled in post-2026 (spec 7.7)
})
print(attribution)
print(proxy_vs_real)

## Summary

- The MVM is evaluated with leave-one-tournament-out (spec 7.1) and judged against the
  RPS < 0.21 / Brier < 0.22 / log loss < 1.02 targets (spec 7.2).
- Calibration curves verify the diagonal fit; the market-feature ablation quantifies market
  information value against the 0.003 RPS kill criterion (spec 7.5).
- Contextual-layer attribution and Tier B proxy-vs-real comparison are completed *after* the
  2026 tournament (spec 7.7-7.8); their schemas are scaffolded here.

Once real artifacts exist (`polymbappe ingest`, `polymbappe train`, `polymbappe backtest`),
every cell above runs on real data with no edits — the synthetic branches are inert.